# EDA: 台灣美食評論情感分類
Exploratory Data Analysis for Taiwan food review sentiment classification.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import jieba
from collections import Counter

matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC', 'Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# Load data — switch to raw/reviews.csv when real data is ready
DATA_PATH = '../data/raw/reviews.csv'  # or '../data/dummy/reviews.csv'
df = pd.read_csv(DATA_PATH)
print(f'Total reviews: {len(df)}')
df.head()

## 1. Label Distribution

In [ ]:
label_names = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
df['label_name'] = df['label'].map(label_names)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
sns.countplot(data=df, x='label_name', order=['Negative', 'Neutral', 'Positive'],
              palette='Set2', ax=axes[0])
axes[0].set_title('Label Distribution', fontsize=13)
axes[0].set_xlabel('Sentiment', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=11)

# Rating distribution
sns.countplot(data=df, x='rating', palette='coolwarm', ax=axes[1])
axes[1].set_title('Rating Distribution', fontsize=13)
axes[1].set_xlabel('Stars', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)

plt.tight_layout()
plt.savefig('../results/figures/eda_label_distribution.png', dpi=150)
plt.show()

print(df['label_name'].value_counts())

## 2. Review Length Distribution

In [ ]:
df['review_length'] = df['review_text'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
df['review_length'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Review Length Distribution', fontsize=13)
axes[0].set_xlabel('Characters', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)

# Box plot by label
sns.boxplot(data=df, x='label_name', y='review_length',
            order=['Negative', 'Neutral', 'Positive'],
            palette='Set2', ax=axes[1])
axes[1].set_title('Review Length by Sentiment', fontsize=13)
axes[1].set_xlabel('Sentiment', fontsize=12)
axes[1].set_ylabel('Characters', fontsize=12)

plt.tight_layout()
plt.savefig('../results/figures/eda_review_length.png', dpi=150)
plt.show()

print(df.groupby('label_name')['review_length'].describe().round(1))

## 3. Top Frequent Words by Sentiment

In [ ]:
from preprocess import tokenize

def get_top_words(texts, top_n=20):
    all_words = []
    for text in texts:
        all_words.extend(tokenize(text))
    return Counter(all_words).most_common(top_n)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for idx, (label, name) in enumerate(label_names.items()):
    texts = df[df['label'] == label]['review_text'].tolist()
    top = get_top_words(texts, 15)
    words, counts = zip(*top)
    axes[idx].barh(range(len(words)), counts, color=['salmon', 'khaki', 'lightgreen'][idx])
    axes[idx].set_yticks(range(len(words)))
    axes[idx].set_yticklabels(words, fontsize=11)
    axes[idx].invert_yaxis()
    axes[idx].set_title(f'Top Words — {name}', fontsize=13)
    axes[idx].set_xlabel('Frequency', fontsize=11)

plt.tight_layout()
plt.savefig('../results/figures/eda_top_words.png', dpi=150)
plt.show()

## 4. Restaurant & Area Coverage

In [ ]:
print(f"Unique restaurants: {df['restaurant'].nunique()}")
print(f"Unique areas: {df['area'].nunique()}")
print()

# Reviews per restaurant (top 20)
top_restaurants = df['restaurant'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(10, 5))
top_restaurants.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Reviews per Restaurant (Top 20)', fontsize=13)
ax.set_xlabel('Number of Reviews', fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../results/figures/eda_restaurants.png', dpi=150)
plt.show()

# Area distribution
if 'area' in df.columns:
    print('\nArea distribution:')
    print(df['area'].value_counts())

## 5. Likes vs Rating

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x='rating', y='likes', palette='coolwarm', ax=ax)
ax.set_title('Likes Distribution by Rating', fontsize=13)
ax.set_xlabel('Star Rating', fontsize=12)
ax.set_ylabel('Likes', fontsize=12)
plt.tight_layout()
plt.savefig('../results/figures/eda_likes_rating.png', dpi=150)
plt.show()

## 6. Dataset Summary

In [ ]:
summary = {
    'Total reviews': len(df),
    'Unique restaurants': df['restaurant'].nunique(),
    'Avg review length (chars)': df['review_length'].mean(),
    'Median review length': df['review_length'].median(),
    'Positive (%)': (df['label'] == 2).mean() * 100,
    'Neutral (%)': (df['label'] == 1).mean() * 100,
    'Negative (%)': (df['label'] == 0).mean() * 100,
}
for k, v in summary.items():
    print(f'{k}: {v:.1f}' if isinstance(v, float) else f'{k}: {v}')